# 🌽 Fine-Tuning — Traductor Mixteco de San Miguel El Grande

Este notebook entrena un modelo de IA para traducir entre Español y Mixteco (Tu'un Savi) de San Miguel El Grande, Oaxaca.

**Pasos:**
1. Instalar dependencias
2. Cargar el dataset `mixteco_dataset.jsonl`
3. Cargar el modelo base (Llama 3.2 3B)
4. Aplicar Fine-Tuning con LoRA
5. Probar el modelo entrenado
6. Guardar el modelo en Google Drive

> **Requisito**: Activa una GPU en Colab → Entorno de ejecución → Cambiar tipo de entorno → GPU (T4 gratis)

## Paso 1 — Instalar dependencias

In [ ]:
!pip install -q unsloth
!pip install -q transformers datasets peft trl bitsandbytes accelerate

## Paso 2 — Subir tu dataset desde Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ⚠️  Cambia esta ruta si guardas el archivo en otra carpeta de tu Drive
DATASET_PATH = '/content/drive/MyDrive/mixteco_dataset.jsonl'

import os
if os.path.exists(DATASET_PATH):
    print(f'✅ Dataset encontrado: {DATASET_PATH}')
else:
    print('❌ No se encontró el dataset. Sube mixteco_dataset.jsonl a tu Google Drive.')

## Paso 3 — Cargar el dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
print(f'Total de ejemplos de entrenamiento: {len(dataset)}')
print('\nEjemplo del dataset:')
print(dataset[0])

## Paso 4 — Cargar el modelo base con LoRA (4-bit quantization)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Modelo base — puedes cambiar por 'mistralai/Mistral-7B-v0.1' si prefieres
BASE_MODEL = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=1024,
    dtype=None,        # Detecta automáticamente float16/bfloat16
    load_in_4bit=True, # Ahorra memoria GPU
)
print('✅ Modelo base cargado.')

## Paso 5 — Configurar LoRA (el módulo de aprendizaje de Mixteco)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # Rango del adaptador LoRA
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('✅ Adaptadores LoRA configurados.')

## Paso 6 — Preparar los datos en formato de instrucción (Alpaca style)

In [ ]:
ALPACA_PROMPT = """A continuación hay una instrucción que describe una tarea de traducción.
Escribe una respuesta que complete correctamente la solicitud.

### Instrucción:
{}

### Entrada:
{}

### Respuesta:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatear_prompt(ejemplos):
    instrucciones = ejemplos['instruction']
    entradas      = ejemplos['input']
    salidas       = ejemplos['output']
    textos = []
    for inst, inp, sal in zip(instrucciones, entradas, salidas):
        texto = ALPACA_PROMPT.format(inst, inp, sal) + EOS_TOKEN
        textos.append(texto)
    return {'text': textos}

dataset_formateado = dataset.map(formatear_prompt, batched=True)
print('✅ Dataset formateado.')
print(dataset_formateado[0]['text'][:400])

## Paso 7 — Entrenar el modelo (Fine-Tuning)

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_formateado,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=1024,
        dataset_num_proc=1,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/content/mixteco_checkpoints",
    ),
)

print('🚀 Iniciando entrenamiento...')
trainer_stats = trainer.train()
print('✅ Entrenamiento completado!')
print(f'   Tiempo total: {trainer_stats.metrics["train_runtime"]:.0f} segundos')
print(f'   Loss final:   {trainer_stats.metrics["train_loss"]:.4f}')

## Paso 8 — Probar el modelo entrenado

In [ ]:
FastLanguageModel.for_inference(model)

def traducir(texto_español):
    prompt = ALPACA_PROMPT.format(
        'Traduce la siguiente oración al Mixteco de San Miguel El Grande (Tu\'un Savi).',
        texto_español,
        ''
    )
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    resultado = tokenizer.batch_decode(outputs)
    # Extraer solo la parte de la Respuesta
    respuesta = resultado[0].split('### Respuesta:')[-1].strip()
    respuesta = respuesta.replace(tokenizer.eos_token, '').strip()
    return respuesta

# Prueba con oraciones de ejemplo
oraciones_prueba = [
    'Buenos días.',
    'El agua está fría.',
    'Los niños juegan en el cerro.',
    'Ella come tortillas con frijoles.',
]

print('=== PRUEBAS DE TRADUCCIÓN ===')
for oración in oraciones_prueba:
    traduccion = traducir(oración)
    print(f'\n  Español: {oración}')
    print(f'  Mixteco: {traduccion}')

## Paso 9 — Guardar el modelo en Google Drive

In [ ]:
SAVE_PATH = '/content/drive/MyDrive/mixteco_lora_model'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f'✅ Modelo guardado en Google Drive: {SAVE_PATH}')
print('   Ahora puedes descargar esta carpeta y usarla en tu servidor.')